# SDXL LoRA Training for Face Replacement

This notebook trains a LoRA on your photos using kohya_ss for SDXL.

**Prerequisites:**
- Upload your `LoRA Photos/train/` folder to Google Drive
- Use Colab Pro with A100 GPU runtime

**Training Config:**
- 69 subject images (wesley_kamau)
- 100 regularization images (person)
- ~2800-3200 steps target, hard stop at 3500

In [7]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

KeyboardInterrupt: 

In [4]:
# Check GPU
!nvidia-smi

Fri Dec 19 09:30:42 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             52W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# Clone kohya_ss sd-scripts
%cd /content
!git clone https://github.com/kohya-ss/sd-scripts.git
%cd sd-scripts
!git checkout sdxl

In [ ]:
# Install dependencies
!pip install -r requirements.txt
!pip install xformers==0.0.23 bitsandbytes==0.41.1 accelerate==0.25.0

In [ ]:
# Download SDXL base model (if not using HF cache)
!pip install huggingface_hub
from huggingface_hub import snapshot_download

# Download to Colab local storage for faster access
model_path = snapshot_download(
    repo_id="stabilityai/stable-diffusion-xl-base-1.0",
    local_dir="/content/sdxl-base",
)
print(f"Model downloaded to: {model_path}")

## 2. Configure Paths

In [ ]:
import os
from pathlib import Path

# === CONFIGURE THESE PATHS ===
# Adjust based on where you uploaded your dataset in Google Drive

DRIVE_BASE = "/content/drive/MyDrive"
DATASET_FOLDER = "LoRA Photos"  # Your folder name in Google Drive

# Paths
TRAIN_DATA_DIR = f"{DRIVE_BASE}/{DATASET_FOLDER}/train"
OUTPUT_DIR = f"{DRIVE_BASE}/{DATASET_FOLDER}/output"
MODEL_PATH = "/content/sdxl-base"  # Local for speed, or use HF path

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify paths exist
print("Checking paths...")
print(f"Train data: {TRAIN_DATA_DIR}")
print(f"  - wesley/: {Path(TRAIN_DATA_DIR, 'wesley').exists()}")
print(f"  - person/: {Path(TRAIN_DATA_DIR, 'person').exists()}")
print(f"Output: {OUTPUT_DIR}")

# Count images
wesley_count = len(list(Path(TRAIN_DATA_DIR, 'wesley').glob('*.jpg')))
person_count = len(list(Path(TRAIN_DATA_DIR, 'person').glob('*.png')))
print(f"\nDataset: {wesley_count} subject images, {person_count} regularization images")

## 3. Create Training Config

In [ ]:
# Training parameters based on loratraining.md recommendations

CONFIG = {
    # Model
    "pretrained_model_name_or_path": MODEL_PATH,
    
    # LoRA network settings
    "network_module": "networks.lora",
    "network_dim": 32,        # Network rank - 32 is good for faces
    "network_alpha": 16,      # Half of dim is common
    
    # Training settings
    "learning_rate": 1e-4,
    "lr_scheduler": "cosine",
    "lr_warmup_steps": 100,
    "train_batch_size": 4,    # A100 can handle 4-6
    "max_train_steps": 3200,  # Target 2800-3200, hard stop at 3500
    "resolution": "1024,1024",
    
    # Optimization
    "optimizer_type": "AdamW8bit",
    "mixed_precision": "fp16",
    "gradient_checkpointing": True,
    
    # Anti-overfitting
    "min_snr_gamma": 5.0,     # Prevents overfitting
    "noise_offset": 0.05,     # Prevents plastic faces
    
    # Saving
    "save_every_n_steps": 500,
    "save_model_as": "safetensors",
    
    # SDXL specific
    "cache_latents": True,
    "cache_latents_to_disk": True,
    "enable_bucket": True,
    "bucket_no_upscale": True,
    "min_bucket_reso": 512,   # Avoid tiny crops and face distortion
    "max_bucket_reso": 2048,
}

# Calculate effective steps
# 69 images × 5 repeats = 345 effective samples per epoch
# With batch size 4: ~86 steps per epoch
# 3200 steps ≈ 37 epochs
print("Training Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Create dataset config TOML file for kohya_ss

dataset_config = f"""
[general]
shuffle_caption = true
caption_extension = ".txt"
keep_tokens = 1

[[datasets]]
resolution = 1024
batch_size = {CONFIG['train_batch_size']}
enable_bucket = true
bucket_no_upscale = true

  [[datasets.subsets]]
  image_dir = "{TRAIN_DATA_DIR}/wesley"
  num_repeats = 5
  class_tokens = "wesley_kamau"

  [[datasets.subsets]]
  image_dir = "{TRAIN_DATA_DIR}/person"
  num_repeats = 1
  is_reg = true
  class_tokens = "person"
"""

config_path = "/content/dataset_config.toml"
with open(config_path, "w") as f:
    f.write(dataset_config)

print(f"Dataset config saved to {config_path}")
print("\n" + dataset_config)

## 4. Generate Caption Files for Regularization Images

The regularization images need caption files with just "person"

In [ ]:
# Create caption files for regularization images (if they don't exist)
from pathlib import Path

reg_dir = Path(TRAIN_DATA_DIR) / "person"
reg_images = list(reg_dir.glob("*.png")) + list(reg_dir.glob("*.jpg"))

created = 0
for img_path in reg_images:
    txt_path = img_path.with_suffix(".txt")
    if not txt_path.exists():
        txt_path.write_text("person")
        created += 1

print(f"Created {created} caption files for regularization images")
print(f"Total regularization images: {len(reg_images)}")

## 5. Start Training

In [ ]:
# Build training command
%cd /content/sd-scripts

train_cmd = f"""
accelerate launch --num_cpu_threads_per_process=2 sdxl_train_network.py \
  --pretrained_model_name_or_path="{CONFIG['pretrained_model_name_or_path']}" \
  --dataset_config="{config_path}" \
  --output_dir="{OUTPUT_DIR}" \
  --output_name="wesley_kamau_lora" \
  --network_module="{CONFIG['network_module']}" \
  --network_dim={CONFIG['network_dim']} \
  --network_alpha={CONFIG['network_alpha']} \
  --learning_rate={CONFIG['learning_rate']} \
  --lr_scheduler="{CONFIG['lr_scheduler']}" \
  --lr_warmup_steps={CONFIG['lr_warmup_steps']} \
  --max_train_steps={CONFIG['max_train_steps']} \
  --optimizer_type="{CONFIG['optimizer_type']}" \
  --mixed_precision="{CONFIG['mixed_precision']}" \
  --save_every_n_steps={CONFIG['save_every_n_steps']} \
  --save_model_as="{CONFIG['save_model_as']}" \
  --min_snr_gamma={CONFIG['min_snr_gamma']} \
  --noise_offset={CONFIG['noise_offset']} \
  --cache_latents \
  --cache_latents_to_disk \
  --gradient_checkpointing \
  --xformers \
  --seed=42
"""

print("Training command:")
print(train_cmd)

In [ ]:
# Run training!
# This will take 6-8 hours for 3200 steps
!{train_cmd}

## 6. Test the Trained LoRA

In [ ]:
# Quick test with the trained LoRA
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
import torch

# Find the latest checkpoint
from pathlib import Path
checkpoints = sorted(Path(OUTPUT_DIR).glob("*.safetensors"))
if checkpoints:
    latest_lora = str(checkpoints[-1])
    print(f"Using LoRA: {latest_lora}")
else:
    print("No checkpoints found yet!")
    latest_lora = None

In [ ]:
# Load pipeline and generate test image
if latest_lora:
    pipe = StableDiffusionXLPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True
    ).to("cuda")
    
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    
    # Load LoRA weights (SDXL-specific syntax)
    import os
    pipe.load_lora_weights(
        os.path.dirname(latest_lora),
        weight_name=os.path.basename(latest_lora)
    )
    
    # Test prompts
    test_prompts = [
        "wesley_kamau, person, male, professional headshot, soft lighting, instagram profile photo style",
        "wesley_kamau, person, male, casual portrait, natural lighting, warm smile",
        "wesley_kamau, person, male, studio portrait, dramatic lighting, looking at camera"
    ]
    
    for i, prompt in enumerate(test_prompts):
        image = pipe(
            prompt=prompt,
            negative_prompt="blurry, distorted, deformed, bad anatomy, ugly",
            num_inference_steps=30,
            guidance_scale=7.5,
            width=1024,
            height=1024,
        ).images[0]
        
        output_path = f"{OUTPUT_DIR}/test_{i+1}.png"
        image.save(output_path)
        print(f"Saved: {output_path}")
        display(image)

## 7. Download Results

Your trained LoRA files are saved in Google Drive at:
`{DRIVE_BASE}/{DATASET_FOLDER}/output/`

Files:
- `wesley_kamau_lora.safetensors` - Final LoRA
- `wesley_kamau_lora-step{N}.safetensors` - Checkpoints every 500 steps

### Next Steps:
1. Download the `.safetensors` file
2. Use with ComfyUI, Automatic1111, or other SDXL-compatible tools
3. Test with prompts starting with `wesley_kamau, person, male, ...`

In [ ]:
# List all output files
!ls -la "{OUTPUT_DIR}"